# 04. EuroCrops collection

Use the local 2021 parcel declarations to collect annual Sentinel-2 imagery for development. CropHarvest remains the reserved test source. Run the relevant source top-ups, then run collection 08 and preprocessing 01–03.

This notebook keeps the existing reference and imagery paths, campaign IDs, top-up settings, and progress logs. The crop-specific setup applies to new requests; earlier batches retain their saved plans. Organized outputs are linked under `pipeline_outputs/collection/eurocrops/`.


## 1. Setup and configuration

Review the local parcel path, batch size, and collection mode before running.

In [11]:
# Review these settings before Run All.
import os
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(root / "src"))
from crop_pipeline import notebook_setup

run_satellite_collection = True
submit_only = False  # Queue exports. Set False to collect finished results.
extra_batches = 21  # Total extra batch slots for this round.
parcels_per_batch = 100
parcel_path = root / "data/external/eurocrops_v2/NL_2021.parquet"
notebook_setup.prepare_environment(root, install_missing=run_satellite_collection)
os.chdir(root)
settings_path = root / "settings.local.json"
settings = notebook_setup.ensure_local_settings(settings_path)
report_root = (root / Path(settings.get("report_root", "data/pipeline_work/inventory")).expanduser()).resolve()


Notebook environment ready: /Users/josedavid/Documents/programming/external-crop-data-classification/.venv/bin/python


## 2. Obtain references

Use the local Dutch EuroCrops parcel declarations for 2021.

In [12]:
from crop_pipeline.sources import adaptive, annual, eurocrops
from crop_pipeline.windows.seasons import file_sha256

print("Local EuroCrops parcel file:", parcel_path)


Local EuroCrops parcel file: /Users/josedavid/Documents/programming/external-crop-data-classification/data/external/eurocrops_v2/NL_2021.parquet


## 3. Validate references

Select declared carrot production parcels with valid footprints.

In [13]:
features = eurocrops.load_carrots(parcel_path)
print("Eligible declared carrot references:", len(features))


Eligible declared carrot references: 2194


## 4. Prepare the collection round

Resume saved plans and select unused carrot parcels for new batches.

In [14]:
batches = adaptive.prepare_round_batches(
    eurocrops.SOURCE, features, annual.archive_root(settings, report_root, repository_root=root), report_root,
    provenance={"dataset": "EuroCrops", "country": "NL", "year": eurocrops.YEAR,
                "parcel_file_sha256": file_sha256(parcel_path),
                "carrot_production_codes": sorted(eurocrops.CARROT_PRODUCTION_CODES)},
    parcels_per_crop=parcels_per_batch, candidate_crops=("carrot",), extra_batches=extra_batches,
)


Round 2: resuming saved eurocrops selections and original collection plans.
Round 2: up to 100 candidates per crop in each new batch. Resume/submit 2 batches.


## 5. Collect annual imagery

Use `submit_only = True` to queue exports. Use `False` to download finished results.

In [15]:
from crop_pipeline import source_workflow

project = notebook_setup.project_for_collection(settings, settings_path, required=run_satellite_collection)
result = adaptive.collect_batches(
    batches, project, enabled=run_satellite_collection, authenticate=run_satellite_collection,
    submit_only=submit_only, download_workers=2, intervals_per_request=6,
)


eurocrops 2021: selecting tiles 1–120 of 120
Scene inventories: 25/120 (0.0s)
Scene inventories: 50/120 (0.0s)
Scene inventories: 75/120 (0.1s)
Scene inventories: 100/120 (0.1s)
Scene inventories: 120/120 (0.1s)
  Resuming with 26,000 rows from 26 verified cached pages.
  Fetching page 27 (up to 5,000 rows; 26,000 saved so far)…
  Page 27 verified: 1,000 rows; 27,000 total (0.6s).
  Fetching page 28 (up to 5,000 rows; 27,000 saved so far)…
  Page 28 verified: 1,000 rows; 28,000 total (0.5s).
  Fetching page 29 (up to 5,000 rows; 28,000 saved so far)…
  Page 29 verified: 1,000 rows; 29,000 total (1.1s).
  Fetching page 30 (up to 5,000 rows; 29,000 saved so far)…
  Page 30 verified: 1,000 rows; 30,000 total (1.5s).
  Fetching page 31 (up to 5,000 rows; 30,000 saved so far)…
  Page 31 verified: 1,000 rows; 31,000 total (0.8s).
  Fetching page 32 (up to 5,000 rows; 31,000 saved so far)…
  Page 32 verified: 1,000 rows; 32,000 total (1.1s).
  Fetching page 33 (up to 5,000 rows; 32,000 saved 

## 6. Summary and next action

Check the batch status below. Once collection is complete, run collection 08, then preprocessing 01–03 to include EuroCrops in the development features.

In [16]:
source_workflow.print_summary(eurocrops.SOURCE, batches, result, "reuse", 10)


eurocrops — completed
References: reuse | Inward buffer: 10 m
Selected parcels: carrot 194
Batches: eurocrops_extra_0021: completed; eurocrops_overnight_100: completed
Verified local tiles: 263/263 (reported batch ranges)
Cloud jobs: 0 queued, 0 running, 0 ready to download, 0 failed, 0 cancelled, 0 unknown
Campaign: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/adaptive_training/829e0cec79b8
Annual TIFF root: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/annual_tifs
Saved registries: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/adaptive_training/829e0cec79b8/rounds/0002/eurocrops_extra_0021/source_handoffs/full_year_all_parcels/eurocrops.json, /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/adaptive_training/829e0cec79b8/rounds/0002/eurocrops_overnight_100/source_handoffs/full_year_all_parcels/eurocrops.json
Next

## Organized outputs

Refresh the linked output folders without moving downloads or changing saved jobs. Run **08_dataset_summary.ipynb** to record all saved batches and selections. Historical outputs above were retained from the original notebook; rerun deliberately to update them.


In [17]:
from crop_pipeline.output_layout import refresh

output_layout = refresh(root)
print("Organized collection outputs:", root / "pipeline_outputs/collection")
print("Next: data_collection/08_dataset_summary.ipynb, then preprocessing/01_imagery_quality.ipynb.")


Organized collection outputs: /Users/josedavid/Documents/programming/external-crop-data-classification/pipeline_outputs/collection
Next: data_collection/06_dataset_summary.ipynb, then preprocessing/01_imagery_quality.ipynb.


## Evidence and interpretation

Counts describe selected references and collection progress. Imagery and window checks determine usable samples. See [research basis and limits](../../docs/research_basis.md) for sources and project-specific choices.
